In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Clone the repository and enter the directory
!git clone https://github.com/sanghyun-son/EDSR-PyTorch
%cd EDSR-PyTorch

# Install dependencies
!pip install imageio scikit-image

Mounted at /content/drive
Cloning into 'EDSR-PyTorch'...
remote: Enumerating objects: 806, done.
remote: Total 806 (delta 0), reused 0 (delta 0), pack-reused 806 (from 1)
Receiving objects: 100% (806/806), 63.09 MiB | 26.99 MiB/s, done.
Resolving deltas: 100% (516/516), done.
/content/EDSR-PyTorch


Paveiksliuku sumazinimas

In [ ]:
import os
from PIL import Image

def generate_low_res_dataset(scale: int):
    # Your exact Drive paths
    low_dir = "/content/drive/MyDrive/EDSR/dataset/LowLight/low"

    # We dynamically name the folder so you can keep x3 and x4 datasets separate!
    lr_dir = f"/content/drive/MyDrive/EDSR/dataset/LowLight/LR_x{scale}"
    os.makedirs(lr_dir, exist_ok=True)

    print(f"Downscaling original DARK images by {scale}x and saving them to: {lr_dir}...")

    for file in os.listdir(low_dir):
        if not file.endswith(('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG', '.NEF')):
            continue

        dark_img = Image.open(os.path.join(low_dir, file))

        # Dynamically shrink based on the scale parameter
        new_width = dark_img.width // scale
        new_height = dark_img.height // scale

        resized_img = dark_img.resize((new_width, new_height), Image.Resampling.BICUBIC)
        resized_img.save(os.path.join(lr_dir, file))

    print(f" Finished creating x{scale} sized dark images in: {lr_dir}")

# Now you just run this command to generate your dataset!
generate_low_res_dataset(4)

Downscaling original DARK images by 4x and saving them to: /content/drive/MyDrive/EDSR/dataset/LowLight/LR_x4...
🎉 Finished creating x4 sized dark images in: /content/drive/MyDrive/EDSR/dataset/LowLight/LR_x4


In [3]:
%cd /content/EDSR-PyTorch

/content/EDSR-PyTorch


In [4]:
import os

content = """import os
from data import srdata
import glob

class LowLight(srdata.SRData):
    def __init__(self, args, name='LowLight', train=True, benchmark=False):
        super(LowLight, self).__init__(
            args, name=name, train=train, benchmark=benchmark
        )

    def _set_filesystem(self, dir_data):
        self.apath = dir_data
        self.dir_hr = os.path.join(self.apath, 'HR')

        # >> FIX: Dynamically link to LR_x3 or LR_x4 depending on your arguments! <<
        scale_val = self.scale[0] if isinstance(self.scale, list) else self.scale
        self.dir_lr = os.path.join(self.apath, f'LR_x{scale_val}')
        self.ext = ('.png', '.png')

    def _scan(self):
        extensions = ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']
        names_hr = []
        for ext in extensions:
            names_hr.extend(glob.glob(os.path.join(self.dir_hr, '*' + ext)))
        names_hr = sorted(names_hr)

        if len(names_hr) == 0:
            print(f"\\n!!! WARNING !!! No images found in: {self.dir_hr}")

        names_lr = [[] for _ in self.scale]
        for f in names_hr:
            filename = os.path.splitext(os.path.basename(f))[0]
            for si, s in enumerate(self.scale):
                found = False
                for ext in extensions:
                    target = os.path.join(self.dir_lr, filename + ext)
                    if os.path.exists(target):
                        names_lr[si].append(target)
                        found = True
                        break
                if not found:
                    names_lr[si].append(os.path.join(self.dir_lr, filename + '.png'))

        return names_hr, names_lr
"""

path = 'src/data/lowlight.py'
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, 'w') as f:
    f.write(content)

print(" Custom Multi-Scale LowLight dataset class created.")


 Custom Multi-Scale LowLight dataset class created.


In [5]:
import os

content = """import random
import numpy as np
import skimage.color as sc
import torch

def get_patch(*args, patch_size=96, scale=2, multi=False, input_large=False):
    ih, iw = args[0].shape[:2]
    if not input_large:
        p = scale if multi else 1
        tp = p * patch_size
        ip = tp // scale
    else:
        tp = patch_size
        ip = patch_size
    ix = random.randrange(0, iw - ip + 1)
    iy = random.randrange(0, ih - ip + 1)
    if not input_large:
        tx, ty = scale * ix, scale * iy
    else:
        tx, ty = ix, iy
    ret = [
        args[0][iy:iy + ip, ix:ix + ip, :],
        *[a[ty:ty + tp, tx:tx + tp, :] for a in args[1:]]
    ]
    return ret

def set_channel(*args, n_channels=3):
    def _set_channel(img):
        if img.ndim == 2:
            img = np.expand_dims(img, axis=2)
        c = img.shape[2]
        if n_channels == 1 and c == 3:
            img = np.expand_dims(sc.rgb2ycbcr(img)[:, :, 0], 2)
        elif n_channels == 3 and c == 1:
            img = np.concatenate([img] * n_channels, 2)
        elif n_channels == 3 and c == 4:
            img = img[:, :, :3]
        return img
    return [_set_channel(a) for a in args]

def np2Tensor(*args, rgb_range=255):
    def _np2Tensor(img):
        np_transpose = np.ascontiguousarray(img.transpose((2, 0, 1)))
        tensor = torch.from_numpy(np_transpose.copy()).float()
        tensor.mul_(rgb_range / 255)
        return tensor
    return [_np2Tensor(a) for a in args]

def augment(*args, hflip=True, rot=True):
    hflip = hflip and random.random() < 0.5
    vflip = rot and random.random() < 0.5
    rot90 = rot and random.random() < 0.5
    def _augment(img):
        if hflip: img = img[:, ::-1, :]
        if vflip: img = img[::-1, :, :]
        if rot90: img = img.transpose(1, 0, 2)
        return img
    return [_augment(a) for a in args]
"""

with open('src/data/common.py', 'w') as f:
    f.write(content)
print(" common.py PyTorch math bugs fixed.")


 common.py PyTorch math bugs fixed.


In [6]:
cd src

/content/EDSR-PyTorch/src


In [ ]:
!python main.py \
    --ext img \
    --model EDSR \
    --scale 3 \
    --shift_mean False \
    --test_every 30 \
    --print_every 10 \
    --patch_size 96 \
    --n_feats 64 \
    --res_scale 0.1 \
    --n_threads 2 \
    --save my_lowlight_x3_model \
    --data_train LowLight \
    --data_test LowLight \
    --dir_data /content/drive/MyDrive/EDSR/dataset/LowLight
    #--epochs 300 \


Making model...
Preparing loss function:
1.000 * L1
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:746: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 1]	Learning rate: 1.00e-4
[160/485]	[L1: 47.6891]	2.0+17.2s
[320/485]	[L1: 47.0507]	0.3+1.8s
[480/485]	[L1: 47.4063]	0.3+1.7s

Evaluation:
100%|█████████████████████████████████████████| 485/485 [00:13<00:00, 34.98it/s]
[LowLight x3]	PSNR: 13.616 (Best: 13.616 @epoch 1)
Forward: 13.87s

Saving...
Total: 14.97s

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:746: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 2]	Learning rate: 1.00e-4
[160/485]	[L1: 47.1028]	0.3+1.9s
[320/485]	[L1: 46.8754]	0.3+1.9s
[480/485]	[L1: 45.2997]	0.4+2.1s

Evaluation:
100%|█████████████████████████████████████████| 485/485 [00:1

In [ ]:
!ls

drive  sample_data


In [6]:
!python main.py \
    --ext img \
    --model EDSR \
    --scale 3 \
    --rgb_range 1 \
    --shift_mean False \
    --test_every 30 \
    --print_every 10 \
    --patch_size 96 \
    --n_feats 64 \
    --res_scale 0.1 \
    --n_threads 2 \
    --save my_lowlight_x3_model_norm \
    --data_train LowLight \
    --data_test LowLight \
    --dir_data /content/drive/MyDrive/EDSR/dataset/LowLight
    #--epochs 300 \


Making model...
Preparing loss function:
1.000 * L1
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:746: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 1]	Learning rate: 1.00e-4
[160/485]	[L1: 0.1884]	1.5+111.0s
[320/485]	[L1: 0.1843]	0.3+74.5s
[480/485]	[L1: 0.1842]	0.3+74.5s

Evaluation:
100%|█████████████████████████████████████████| 485/485 [00:15<00:00, 31.91it/s]
[LowLight x3]	PSNR: 14.042 (Best: 14.042 @epoch 1)
Forward: 15.21s

Saving...
Total: 16.30s

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:746: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
[Epoch 2]	Learning rate: 1.00e-4
[160/485]	[L1: 0.1781]	0.4+1.7s
[320/485]	[L1: 0.1742]	0.3+1.6s
[480/485]	[L1: 0.1661]	0.4+1.8s

Evaluation:
100%|█████████████████████████████████████████| 485/485 [00:14<0

In [7]:
# Create a folder in your Drive to hold your custom models
!mkdir -p /content/drive/MyDrive/EDSR_models

# Copy the trained experiment folder into your Drive
!cp -r /content/EDSR-PyTorch/experiment/my_lowlight_x3_model_norm /content/drive/MyDrive/EDSR_models/

print(" Model successfully backed up to Google Drive!")


 Model successfully backed up to Google Drive!


In [ ]:
!ls

data	       demo.sh	    loss     model	template.py  utility.py
dataloader.py  __init__.py  main.py  option.py	trainer.py   videotester.py


In [9]:
cd src

/content/EDSR-PyTorch/src


In [9]:
TEST_IMAGES_DIR = "/content/drive/MyDrive/EDSR/data_test/lowLight/low"
MODEL_PATH = "/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/model/model_latest.pt"
SCALE = 3
SAVE_NAME = "my_test_results_x3_LOL"

!python main.py \
    --data_test Demo \
    --dir_demo "{TEST_IMAGES_DIR}" \
    --model EDSR \
    --scale {SCALE} \
    --n_feats 64 \
    --res_scale 0.1 \
    --shift_mean False \
    --pre_train "{MODEL_PATH}" \
    --test_only \
    --save_results \
    --save "{SAVE_NAME}"
    #--rgb_range 1 \


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Making model...
Load the model from /content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/model/model_latest.pt

Evaluation:
  0%|                                                    | 0/15 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation

In [3]:
%cd /content/EDSR-PyTorch

/content/EDSR-PyTorch


In [31]:
from PIL import Image
import os

input_dir = "/content/drive/MyDrive/ExDark/Bicycle"
output_dir = "/content/drive/MyDrive/ExDark/Bicycle_small"
os.makedirs(output_dir, exist_ok=True)

for f in os.listdir(input_dir):
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        img = Image.open(os.path.join(input_dir, f)).convert("RGB")
        img.thumbnail((768, 768))
        img.save(os.path.join(output_dir, f))

In [10]:
!ls

data	       demo.sh	    loss     model	template.py  utility.py
dataloader.py  __init__.py  main.py  option.py	trainer.py   videotester.py


In [12]:
#LOW_LIGHT_DIR = "/content/drive/MyDrive/NPE_small"
LOW_LIGHT_DIR = "/content/drive/MyDrive/DICM"
MODEL_PATH = "/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/model/model_latest.pt"
SCALE = 3
SAVE_NAME = "my_test_results_x3_DICM"

!python main.py \
    --data_test Demo \
    --dir_demo "{LOW_LIGHT_DIR}" \
    --model EDSR \
    --scale {SCALE} \
    --n_feats 64 \
    --res_scale 0.1 \
    --shift_mean False \
    --pre_train "{MODEL_PATH}" \
    --test_only \
    --save_results \
    --save "{SAVE_NAME}" \



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Making model...
Load the model from /content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/model/model_latest.pt

Evaluation:
  0%|                                                    | 0/64 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation

Modelio tikslumo metrikos

In [14]:
!pip install scikit-image opencv-python numpy lpips torch torchvision pyiqa


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [15]:
import os
import cv2
import torch
import numpy as np
import lpips
import pyiqa
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Initialize PyTorch device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize LPIPS loss function (using VGG network, which is the standard)
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

# Initialize No-Reference metrics NIQE and BRISQUE using pyiqa
niqe_metric = pyiqa.create_metric('niqe', device=device)
brisque_metric = pyiqa.create_metric('brisque', device=device)

def compute_gssim(img1, img2):
    """
    Computes Gradient-based SSIM.
    It extracts gradient maps (edges) via Sobel filters and computes SSIM on them.
    """
    # Convert to grayscale for gradient extraction
    img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Calculate gradients using Sobel operators
    gx1 = cv2.Sobel(img1_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy1 = cv2.Sobel(img1_gray, cv2.CV_64F, 0, 1, ksize=3)
    grad1 = cv2.magnitude(gx1, gy1)

    gx2 = cv2.Sobel(img2_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy2 = cv2.Sobel(img2_gray, cv2.CV_64F, 0, 1, ksize=3)
    grad2 = cv2.magnitude(gx2, gy2)

    # Calculate data range dynamically based on gradients
    grad_range = max(grad1.max(), grad2.max()) - min(grad1.min(), grad2.min())
    if grad_range == 0:
        grad_range = 1.0 # Fallback to prevent division by zero

    # Compute structural similarity on the gradient magnitude maps
    score = ssim(grad1, grad2, data_range=grad_range)
    return score

#change the scale here
def evaluate_results(pred_dir, gt_dir=None, scale=3):
    """
    Evaluates results dynamically.
    If gt_dir is provided and exists: Computes both Full-Reference and No-Reference metrics.
    If gt_dir is None or invalid:   Computes only No-Reference metrics (NIQE and BRISQUE) on predictions.
    """
    valid_ext = ['.png', '.jpg', '.jpeg', '.bmp']

    # ---------------------------------------------
    # Case A: No-Reference Mode (No Ground Truth)
    # ---------------------------------------------
    if gt_dir is None or gt_dir.strip() == "" or gt_dir.strip().lower() == "none" or not os.path.exists(gt_dir):
        print("No Ground Truth directory found or provided. Running in NO-REFERENCE mode...")
        pred_files = sorted([f for f in os.listdir(pred_dir) if os.path.splitext(f)[-1].lower() in valid_ext])

        if len(pred_files) == 0:
            print("No images found in prediction directory!")
            return

        print(f"Evaluating {len(pred_files)} predictions...")
        niqe_list, brisque_list = [], []

        for filename in pred_files:
            pred_path = os.path.join(pred_dir, filename)
            with torch.no_grad():
                niqe_score = niqe_metric(pred_path).item()
                brisque_score = brisque_metric(pred_path).item()
            niqe_list.append(niqe_score)
            brisque_list.append(brisque_score)

        print("\n========= Final No-Reference Results =========")
        print(f"Total Images Evaluated: {len(pred_files)}")
        print(f"Average NIQE Score:    {np.mean(niqe_list):.4f} (Lower is Better)")
        print(f"Average BRISQUE Score: {np.mean(brisque_list):.4f} (Lower is Better)")
        print("==============================================")
        return

    # ---------------------------------------------
    # Case B: Full-Reference Mode (Ground Truth Available)
    # ---------------------------------------------
    print(" Ground Truth directory found. Running in FULL-REFERENCE mode...")
    mse_list, rmse_list, mae_list, psnr_list = [], [], [], []
    ssim_list, gssim_list, lpips_list = [], [], []
    niqe_list, brisque_list = [], []

    gt_files = sorted([f for f in os.listdir(gt_dir) if os.path.splitext(f)[-1].lower() in valid_ext])

    if len(gt_files) == 0:
        print("No images found in ground truth directory!")
        return

    print(f"Evaluating {len(gt_files)} images at x{scale} scale...")

    for filename in gt_files:
        gt_path = os.path.join(gt_dir, filename)

        name, ext = os.path.splitext(filename)

        # --- HERE IS THE FIX: DYNAMIC SCALE IN FILENAME ---
        pred_filename = f"{name}_x{scale}_SR{ext}"
        pred_path = os.path.join(pred_dir, pred_filename)

        # Fallback if the SR tag is missing
        if not os.path.exists(pred_path):
            pred_path = os.path.join(pred_dir, filename)
            if not os.path.exists(pred_path):
                print(f"Warning: Prediction for {filename} not found!")
                continue

        gt_img = cv2.imread(gt_path)
        pred_img = cv2.imread(pred_path)

        # Ensure sizes match (Resize prediction if dimensions differ slightly due to rounding)
        if gt_img.shape != pred_img.shape:
            pred_img = cv2.resize(pred_img, (gt_img.shape[1], gt_img.shape[0]), interpolation=cv2.INTER_CUBIC)

        gt_float = gt_img.astype(np.float32)
        pred_float = pred_img.astype(np.float32)

        # --- 1. MSE, RMSE & MAE ---
        mse_val = np.mean((gt_float - pred_float) ** 2)
        mse_list.append(mse_val)
        rmse_list.append(np.sqrt(mse_val))
        mae_list.append(np.mean(np.abs(gt_float - pred_float)))

        # --- 2. PSNR ---
        psnr_list.append(psnr(gt_img, pred_img, data_range=255))

        # --- 3. SSIM ---
        ssim_list.append(ssim(gt_img, pred_img, channel_axis=2, data_range=255))

        # --- 4. GSSIM ---
        gssim_list.append(compute_gssim(gt_img, pred_img))

        # --- 5. LPIPS ---
        # LPIPS expects RGB PyTorch Tensors normalized between [-1, 1]
        gt_rgb = cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB)
        pred_rgb = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)

        # CHW format, add batch dim, normalize to [-1, 1], move to GPU/CPU
        gt_tensor = torch.from_numpy(gt_rgb.transpose(2,0,1)).float().unsqueeze(0) / 127.5 - 1.0
        pred_tensor = torch.from_numpy(pred_rgb.transpose(2,0,1)).float().unsqueeze(0) / 127.5 - 1.0

        gt_tensor = gt_tensor.to(device)
        pred_tensor = pred_tensor.to(device)

        # Calculate LPIPS
        with torch.no_grad():
            lpips_score = loss_fn_vgg(gt_tensor, pred_tensor)
        lpips_list.append(lpips_score.item())

        # --- 6. NIQE & BRISQUE (No-Reference Metrics on prediction) ---
        with torch.no_grad():
            niqe_score = niqe_metric(pred_path).item()
            brisque_score = brisque_metric(pred_path).item()
        niqe_list.append(niqe_score)
        brisque_list.append(brisque_score)

    # -----------------------
    # --- Print Summary ---
    # -----------------------
    avg_mse = np.mean(mse_list)
    avg_rmse = np.mean(rmse_list)
    avg_mae = np.mean(mae_list)
    avg_psnr = np.mean(psnr_list)
    avg_ssim = np.mean(ssim_list)
    avg_gssim = np.mean(gssim_list)
    avg_lpips = np.mean(lpips_list)
    avg_niqe = np.mean(niqe_list)
    avg_brisque = np.mean(brisque_list)

    print("\n========= Final Evaluation Results =========")
    print(f"Total Images Evaluated: {len(mse_list)}\n")

    # MAE as error percentage (out of max possible pixel error of 255)
    mae_percent = (avg_mae / 255.0) * 100
    # SSIM and GSSIM as match percentage (out of max 1.0)
    ssim_percent = avg_ssim * 100
    gssim_percent = avg_gssim * 100
    # LPIPS as perceptual error percentage
    lpips_percent = avg_lpips * 100

    print("--- Similarity / Accuracy Metrics (Higher % is Better) ---")
    print(f"SSIM Match:          {ssim_percent:.2f}% (Structural Accuracy)")
    print(f"GSSIM Match:         {gssim_percent:.2f}% (Edge Fidelity Accuracy)")

    print("\n--- Error / Difference Metrics (Lower % is Better) ---")
    print(f"LPIPS Difference:    {lpips_percent:.2f}% (Deep Perceptual Error)")
    print(f"MAE Pixel Error:     {mae_percent:.2f}% (Average Raw Pixel Error)")
    print(f"Raw MSE Value:       {avg_mse:.2f}")
    print(f"Raw RMSE Value:      {avg_rmse:.2f}")

    print("\n--- Standard Quality Metrics ---")
    print(f"PSNR:                {avg_psnr:.2f} dB (Logarithmic, cannot be %)")

    print("\n--- No-Reference Quality Metrics (Lower is Better Quality) ---")
    print(f"NIQE Score:          {avg_niqe:.4f}")
    print(f"BRISQUE Score:       {avg_brisque:.4f}")
    print("============================================")

# ==========================================
#@title  Metrics Evaluation Config Form
# ==========================================
PREDICTION_DIR = "/content/EDSR-PyTorch/experiment/my_test_results_x3_DICM/results-Demo"
#GROUND_TRUTH_DIR = "/content/drive/MyDrive/EDSR/data_test/ground_truth/high"
GROUND_TRUTH_DIR = None
SCALE = 3

evaluate_results(PREDICTION_DIR, GROUND_TRUTH_DIR, scale=SCALE)


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 70.1MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/niqe_modelparameters.mat" to /root/.cache/torch/hub/pyiqa/niqe_modelparameters.mat



100%|██████████| 8.15k/8.15k [00:00<00:00, 28.3MB/s]

Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/brisque_svm_weights.pth" to /root/.cache/torch/hub/pyiqa/brisque_svm_weights.pth




100%|██████████| 112k/112k [00:00<00:00, 8.96MB/s]


No Ground Truth directory found or provided. Running in NO-REFERENCE mode...
Evaluating 64 predictions...

========= Final No-Reference Results =========
Total Images Evaluated: 64
Average NIQE Score:    6.2635 (Lower is Better)
Average BRISQUE Score: 46.6065 (Lower is Better)


In [ ]:
GROUND_TRUTH_DIR = "/content/drive/MyDrive/EDSR/data_test/ground_truth/high"
PREDICTION_DIR = "/content/EDSR-PyTorch/experiment/my_test_results/results-Demo"


In [37]:
DRIVE_SAVE_PATH = "/content/drive/MyDrive/EDSR/x3_norm_scaled_results"
!mkdir -p "{DRIVE_SAVE_PATH}"
!cp -r "/content/EDSR-PyTorch/experiment/my_test_results_x3_norm/results-Demo/." "{DRIVE_SAVE_PATH}/"

print(" Predictions successfully backed up to your Google Drive!")

 Predictions successfully backed up to your Google Drive!


testing the original model

In [ ]:
#python main.py --data_test Demo --scale 4 --pre_train download --test_only --save_results

In [ ]:
!ls

experiment  figs  LICENSE  README.md  src  test


In [ ]:
cd src

/content/EDSR-PyTorch/src


In [ ]:
!python main.py --model EDSR --scale 3 --patch_size 144 --save edsr_baseline_x3 --reset --pre_train /content/EDSR-PyTorch/experiment/EDSR_x2.pt --dir_data /content/EDSR-PyTorch/test

Traceback (most recent call last):
  File "/content/EDSR-PyTorch/src/main.py", line 33, in <module>
    main()
  File "/content/EDSR-PyTorch/src/main.py", line 22, in main
    loader = data.Data(args)
             ^^^^^^^^^^^^^^^
  File "/content/EDSR-PyTorch/src/data/__init__.py", line 26, in __init__
    self.loader_train = dataloader.DataLoader(
                        ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 394, in __init__
    sampler = RandomSampler(dataset, generator=generator)  # type: ignore[arg-type]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/sampler.py", line 149, in __init__
    raise ValueError(
ValueError: num_samples should be a positive integer value, but got num_samples=0


In [ ]:
#doesn't give results
!python main.py --model EDSR --scale 4 --patch_size 144 --save edsr_baseline_x3 --n_feats 256 --reset --pre_train /content/EDSR-PyTorch/experiment/EDSR_x2.pt --test_only --data_test Demo --dir_demo /content/EDSR-PyTorch/test


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Making model...
Load the model from /content/EDSR-PyTorch/experiment/EDSR_x2.pt

Evaluation:
  0%|                                                     | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow

In [ ]:
!python main.py --model EDSR --scale 4 --n_resblocks 32 --n_feats 256 --res_scale 0.1 --data_test Demo --dir_demo /content/EDSR-PyTorch/test --pre_train /content/EDSR-PyTorch/experiment/EDSR_x4.pt --test_only --save_results #--n_threads 1 --cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Making model...
Load the model from /content/EDSR-PyTorch/experiment/EDSR_x4.pt

Evaluation:
  0%|                                                     | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow

SPECS

In [2]:
!nvidia-smi

Tue May 26 10:10:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.__version__)
print(torch.version.cuda)

Tesla T4
2.10.0+cu128
12.8


In [4]:
import os

exp = "/content/EDSR-PyTorch/experiment"
print(os.listdir(exp))

['.gitignore']


In [5]:
print(os.listdir("/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model"))

['results-LowLight', 'model', 'log.txt', 'config.txt', 'loss.pt', 'loss_log.pt', 'loss_L1.pdf', 'test_LowLight.pdf', 'optimizer.pt', 'psnr_log.pt']


In [7]:
with open("/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/log.txt") as f:
    print(f.read())

EDSR(
  (sub_mean): MeanShift(3, 3, kernel_size=(1, 1), stride=(1, 1))
  (add_mean): MeanShift(3, 3, kernel_size=(1, 1), stride=(1, 1))
  (head): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (body): Sequential(
    (0): ResBlock(
      (body): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (1): ResBlock(
      (body): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (2): ResBlock(
      (body): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
 

In [8]:
with open("/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/config.txt") as f:
    print(f.read())

2026-04-13-20:37:03

debug: False
template: .
n_threads: 2
cpu: False
n_GPUs: 1
seed: 1
dir_data: /content/drive/MyDrive/EDSR/dataset/LowLight
dir_demo: ../test
data_train: ['LowLight']
data_test: ['LowLight']
data_range: 1-800/801-810
ext: img
scale: [3]
patch_size: 96
rgb_range: 255
n_colors: 3
chop: False
no_augment: False
model: EDSR
act: relu
pre_train: 
extend: .
n_resblocks: 16
n_feats: 64
res_scale: 0.1
shift_mean: False
dilation: False
precision: single
G0: 64
RDNkSize: 3
RDNconfig: B
n_resgroups: 10
reduction: 16
reset: False
test_every: 30
epochs: 300
batch_size: 16
split_batch: 1
self_ensemble: False
test_only: False
gan_k: 1
lr: 0.0001
decay: 200
gamma: 0.5
optimizer: ADAM
momentum: 0.9
betas: (0.9, 0.999)
epsilon: 1e-08
weight_decay: 0
gclip: 0
loss: 1*L1
skip_threshold: 100000000.0
save: my_lowlight_x3_model
load: 
resume: 0
save_models: False
print_every: 10
save_results: False
save_gt: False




In [10]:
import torch

ckpt = torch.load(
"/content/drive/MyDrive/EDSR_models/my_lowlight_x3_model/model/model_latest.pt",
map_location="cpu"
)

print(ckpt.keys())

odict_keys(['sub_mean.weight', 'sub_mean.bias', 'add_mean.weight', 'add_mean.bias', 'head.0.weight', 'head.0.bias', 'body.0.body.0.weight', 'body.0.body.0.bias', 'body.0.body.2.weight', 'body.0.body.2.bias', 'body.1.body.0.weight', 'body.1.body.0.bias', 'body.1.body.2.weight', 'body.1.body.2.bias', 'body.2.body.0.weight', 'body.2.body.0.bias', 'body.2.body.2.weight', 'body.2.body.2.bias', 'body.3.body.0.weight', 'body.3.body.0.bias', 'body.3.body.2.weight', 'body.3.body.2.bias', 'body.4.body.0.weight', 'body.4.body.0.bias', 'body.4.body.2.weight', 'body.4.body.2.bias', 'body.5.body.0.weight', 'body.5.body.0.bias', 'body.5.body.2.weight', 'body.5.body.2.bias', 'body.6.body.0.weight', 'body.6.body.0.bias', 'body.6.body.2.weight', 'body.6.body.2.bias', 'body.7.body.0.weight', 'body.7.body.0.bias', 'body.7.body.2.weight', 'body.7.body.2.bias', 'body.8.body.0.weight', 'body.8.body.0.bias', 'body.8.body.2.weight', 'body.8.body.2.bias', 'body.9.body.0.weight', 'body.9.body.0.bias', 'body.9.bo

In [ ]:
model = trainer.model   # if notebook still has model loaded

total = sum(p.numel() for p in model.parameters())
print("Parameters:", total)